# 01 — Data Preparation (Exon 4-15 Strategy)

===========================================
Parse tBLASTn results for MROH6 in the zebra finch genome using two protein
queries combined with exon 13 BLASTn anchors. Apply Tony's exon 4-15 strategy:
filter for gene units with >=50% coverage of the exon 4-15 region to produce
a clean codon alignment for dN/dS analysis.

Strategy change (March 2026):
  OLD: Use all exons, 300bp length filter → 411 gene units, noisy alignment
  NEW: Exon 4-15 focus, 50% coverage filter → cleaner alignment for PAML

Usage:
  python notebooks/01_data_prep.py

In [1]:
import sys
sys.path.insert(0, '../scripts')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from Bio import SeqIO, AlignIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from pathlib import Path
import subprocess

from utils import (
    parse_blast_fasta, parse_exon13_blastn, parse_combined_tblastn,
    build_accession_to_chrom, define_gene_units_exon4_15, gene_units_to_fasta,
    merge_overlapping_hits, loci_to_fasta, classify_chrom,
    EXON4_15_LEN_NT, EXON4_START_AA, EXON15_END_AA
)

PROJECT = Path(__file__).resolve().parent.parent
DATA_RAW = PROJECT / 'data' / 'raw'
DATA_PROC = PROJECT / 'data' / 'processed'
RESULTS = PROJECT / 'results'
INPUT_DIR = PROJECT / 'Zebra finch input files'
FIG_DIR = RESULTS / 'figures'
TABLE_DIR = RESULTS / 'tables'

for d in [DATA_PROC, FIG_DIR, TABLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

sns.set_context('notebook')
sns.set_style('whitegrid')

MIN_COVERAGE = 0.50  # Tony's MIN_PAML_COV threshold

NameError: name '__file__' is not defined

## Summary

In [ ]:
print("=" * 70)
print("  STEP 01: DATA PREPARATION — Exon 4-15 Strategy")

## Summary

In [ ]:
print("=" * 70)

## 1a. Parse inputs

## 1a. Parsing exon 13 anchors and tBLASTn hits

In [ ]:
print("\n── 1a. Parsing exon 13 anchors and tBLASTn hits ──")

exon13_file = INPUT_DIR / 'MROH6 exon 13 V5SBW1G0014-Alignment.txt'
exon13 = parse_exon13_blastn(exon13_file)
print(f"  Exon 13 anchors: {len(exon13)} across {exon13['saccver'].nunique()} accessions")

tblastn_file1 = INPUT_DIR / 'Taeniopygia guttata tBLASTn MROH6 XP_030133788.3 alignment.txt'
tblastn_file2 = INPUT_DIR / 'Taeniopygia guttata tBLASTn MROH6 XP_072787743.1 alignment.txt'

df1 = parse_blast_fasta(tblastn_file1)
df2 = parse_blast_fasta(tblastn_file2)
tblastn = parse_combined_tblastn(tblastn_file1, tblastn_file2)
acc_to_chrom = build_accession_to_chrom(tblastn)

print(f"  tBLASTn hits:")
print(f"    XP_030133788.3 (isoform X2): {len(df1)}")
print(f"    XP_072787743.1 (isoform X1): {len(df2)}")
print(f"    Combined (deduplicated):     {len(tblastn)}")
print(f"    New hits from 2nd query:     {len(tblastn) - len(df1)}")

# Check which accessions lack exon 13
exon13_accs = set(exon13['saccver'].unique())
blast_accs = set(tblastn['accession'].unique())
missing_accs = blast_accs - exon13_accs
print(f"  Accessions without exon 13 anchor:")
for acc in sorted(missing_accs):
    chrom = acc_to_chrom.get(acc, '?')
    n_hits = len(tblastn[tblastn['accession'] == acc])
    print(f"    {acc} (chr {chrom}): {n_hits} tBLASTn hits")

## Figure 1: BLAST hit overview

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

chrom_counts = tblastn['chrom'].value_counts().head(20)
colors = ['crimson' if c == '7' else ('darkorange' if c in ['16','25','29','30','31','33','34','35','36','37'] else 'steelblue')
          for c in chrom_counts.index]
chrom_counts.plot.bar(ax=axes[0], color=colors)
axes[0].set_title('tBLASTn hits per chromosome (combined, deduplicated)')
axes[0].set_xlabel('Chromosome')
axes[0].set_ylabel('Number of hits')

axes[1].hist(tblastn['seq_len'], bins=50, color='steelblue', edgecolor='white')
axes[1].set_title('Individual BLAST hit lengths')
axes[1].set_xlabel('Hit length (bp)')
axes[1].set_ylabel('Count')

macro_chroms = ['1', '1A', '2', '3', '4', '4A', '5', '6', '7', '8']
micro_chroms = [c for c in tblastn['chrom'].unique()
                if c not in macro_chroms and c not in ['Z', 'W', 'unknown']]
cats = {
    'Chr 7\n(ancestral)': len(tblastn[tblastn['chrom'] == '7']),
    'Other macro\n(1-8)': len(tblastn[(tblastn['chrom'].isin(macro_chroms)) & (tblastn['chrom'] != '7')]),
    'Micro\n(9-37)': len(tblastn[tblastn['chrom'].isin(micro_chroms)]),
    'Sex\n(Z/W)': len(tblastn[tblastn['chrom'].isin(['Z', 'W'])]),
}
axes[2].bar(cats.keys(), cats.values(),
            color=['crimson', 'steelblue', 'darkorange', 'mediumpurple'])
axes[2].set_title('Hits: Ancestral chr7 vs Derived')
axes[2].set_ylabel('Number of hits')
for i, (k, v) in enumerate(cats.items()):
    axes[2].text(i, v + 20, str(v), ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(FIG_DIR / 'blast_hit_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"  Saved: {FIG_DIR / 'blast_hit_overview.png'}")

## 1b. Define gene units with exon 4-15 coverage filter

In [ ]:
print(f"\n── 1b. Defining gene units (exon 4-15 strategy, {MIN_COVERAGE*100:.0f}% coverage) ──")
print(f"  Expected exon 4-15 CDS length: {EXON4_15_LEN_NT} nt ({EXON15_END_AA - EXON4_START_AA + 1} aa)")
print(f"  Minimum sequence length: {int(EXON4_15_LEN_NT * MIN_COVERAGE)} nt")

gu_all, gu_filtered = define_gene_units_exon4_15(
    exon13, tblastn, acc_to_chrom,
    max_dist=25000, merge_gap=500, min_coverage=MIN_COVERAGE
)

# Classify chromosomes
gu_all['chrom_class'] = gu_all['chrom'].apply(classify_chrom)
gu_filtered['chrom_class'] = gu_filtered['chrom'].apply(classify_chrom)

# Identify ancestral copy
chr7_gu = gu_filtered[(gu_filtered['chrom'] == '7') &
                       (gu_filtered['start'] > 28_000_000) &
                       (gu_filtered['end'] < 29_500_000)]
if len(chr7_gu) > 0:
    ancestral_idx = chr7_gu['total_seq_len'].idxmax()
    ancestral_id = gu_filtered.loc[ancestral_idx, 'gene_unit_id']
    gu_filtered['is_ancestral'] = gu_filtered['gene_unit_id'] == ancestral_id
    gu_all['is_ancestral'] = gu_all['gene_unit_id'] == ancestral_id
    print(f"  Ancestral copy: gu_{ancestral_id} "
          f"(chr7, span={gu_filtered.loc[ancestral_idx, 'span']}bp)")
else:
    gu_filtered['is_ancestral'] = False
    gu_all['is_ancestral'] = False
    print("  WARNING: Could not identify ancestral locus")

print(f"\n  Gene units defined: {len(gu_all)}")
print(f"    With exon 13 anchor:     {gu_all['has_exon13'].sum()}")
print(f"    Without anchor (Z/20):   {(~gu_all['has_exon13']).sum()}")
print(f"\n  After {MIN_COVERAGE*100:.0f}% coverage filter: {len(gu_filtered)} gene units")
print(f"    Removed: {len(gu_all) - len(gu_filtered)} (insufficient exon 4-15 coverage)")
print(f"\n  Filtered chromosome classification:")
print(f"    {gu_filtered['chrom_class'].value_counts().to_string()}")

## Figure 2: Coverage filter visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Coverage distribution
ax = axes[0]
cov = gu_all['coverage_frac']
ax.hist(cov[cov > 0], bins=50, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(MIN_COVERAGE, color='red', linestyle='--', linewidth=2,
           label=f'{MIN_COVERAGE*100:.0f}% cutoff')
n_pass = (cov >= MIN_COVERAGE).sum()
n_total = (cov > 0).sum()
ax.set_title(f'Exon 4-15 coverage distribution\n'
             f'{n_pass}/{n_total} pass {MIN_COVERAGE*100:.0f}% threshold')
ax.set_xlabel('Fraction of exon 4-15 region covered')
ax.set_ylabel('Count')
ax.legend()

# Before vs after filter
ax = axes[1]
before_class = gu_all[gu_all['total_seq_len'] > 0]['chrom_class'].value_counts()
after_class = gu_filtered['chrom_class'].value_counts()
x = range(len(before_class))
w = 0.35
ax.bar([i - w/2 for i in x], before_class.values, w,
       label='Before filter', color='lightcoral', alpha=0.8)
after_vals = [after_class.get(c, 0) for c in before_class.index]
ax.bar([i + w/2 for i in x], after_vals, w,
       label='After filter', color='steelblue', alpha=0.8)
ax.set_xticks(list(x))
ax.set_xticklabels(before_class.index, rotation=20, ha='right')
ax.set_ylabel('Gene units')
ax.set_title('Gene units before/after coverage filter')
ax.legend()

# Sequence length distribution (filtered)
ax = axes[2]
ax.hist(gu_filtered['total_seq_len'], bins=40, color='steelblue', edgecolor='white')
ax.set_xlabel('Total sequence length (bp)')
ax.set_ylabel('Count')
ax.set_title(f'Filtered gene unit lengths\n(n={len(gu_filtered)})')
ax.axvline(gu_filtered['total_seq_len'].median(), color='red', linestyle='--',
           label=f"Median={gu_filtered['total_seq_len'].median():.0f} bp")
ax.legend()

plt.tight_layout()
plt.savefig(FIG_DIR / 'exon4_15_coverage_filter.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"  Saved: {FIG_DIR / 'exon4_15_coverage_filter.png'}")

## Figure 3: Exon 13 anchor quality

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
ax.hist(exon13['pident'], bins=20, color='seagreen', edgecolor='white')
ax.set_xlabel('% Identity to exon 13 probe')
ax.set_ylabel('Count')
ax.set_title(f'Exon 13 anchor identity (n={len(exon13)})')
ax.axvline(exon13['pident'].median(), color='red', linestyle='--',
           label=f'Median={exon13["pident"].median():.1f}%')
ax.legend()

ax = axes[1]
exon13_chroms = exon13['saccver'].map(acc_to_chrom).fillna('unknown')
chrom_counts_e13 = exon13_chroms.value_counts().head(15)
chrom_counts_e13.plot.bar(ax=ax, color='seagreen')
ax.set_title('Exon 13 anchors per chromosome')
ax.set_xlabel('Chromosome')
ax.set_ylabel('Anchors')

ax = axes[2]
blast_by_chrom = tblastn['chrom'].value_counts()
e13_by_chrom = exon13_chroms.value_counts()
compare_chroms = blast_by_chrom.head(12).index
x = range(len(compare_chroms))
w = 0.35
ax.bar([i - w/2 for i in x],
       [blast_by_chrom.get(c, 0) for c in compare_chroms],
       w, label='tBLASTn hits', color='steelblue', alpha=0.8)
ax.bar([i + w/2 for i in x],
       [e13_by_chrom.get(c, 0) * 10 for c in compare_chroms],
       w, label='Exon 13 anchors (x10)', color='seagreen', alpha=0.8)
ax.set_xticks(list(x))
ax.set_xticklabels(compare_chroms, rotation=45)
ax.set_title('tBLASTn hits vs exon 13 anchors')
ax.legend()

plt.tight_layout()
plt.savefig(FIG_DIR / 'exon13_anchor_quality.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"  Saved: {FIG_DIR / 'exon13_anchor_quality.png'}")

## Figure 4: Genomic distribution

In [ ]:
top_chroms = gu_filtered['chrom'].value_counts().head(8).index.tolist()
for c in ['7', 'Z']:
    if c not in top_chroms and c in gu_filtered['chrom'].values:
        top_chroms.append(c)

fig, axes = plt.subplots(len(top_chroms), 1,
                         figsize=(14, 2 * len(top_chroms)), sharex=False)
if len(top_chroms) == 1:
    axes = [axes]

for ax, chrom in zip(axes, top_chroms):
    chrom_gu = gu_filtered[gu_filtered['chrom'] == chrom]
    positions = chrom_gu['start'].values / 1e6
    color = 'crimson' if chrom == '7' else ('mediumpurple' if chrom == 'Z' else 'darkorange')
    ax.scatter(positions, [1]*len(positions), marker='|', s=100, color=color, alpha=0.7)
    ax.set_yticks([])
    ax.set_ylabel(f'chr {chrom}\n({len(chrom_gu)})', rotation=0, ha='right', va='center')
    if len(positions) > 0:
        ax.set_xlim(positions.min() - 0.5, positions.max() + 0.5)

axes[-1].set_xlabel('Genomic position (Mb)')
fig.suptitle('Gene unit distribution along chromosomes\n'
             '(dispersed = retrotransposition; clustered = tandem duplication)',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'gene_unit_genomic_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"  Saved: {FIG_DIR / 'gene_unit_genomic_distribution.png'}")

## Figure 5: Hits per gene unit + coverage scatter

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.hist(gu_filtered['n_hits'], bins=30, color='steelblue', edgecolor='white')
ax.set_xlabel('tBLASTn hits per gene unit')
ax.set_ylabel('Count')
ax.set_title(f'tBLASTn coverage per gene unit (median={gu_filtered["n_hits"].median():.0f})')

ax = axes[1]
color_map = {'chr7_ancestral': 'crimson', 'macro_derived': 'steelblue',
             'micro_derived': 'darkorange', 'sex_chrom': 'mediumpurple'}
for cls, color in color_map.items():
    mask = gu_filtered['chrom_class'] == cls
    ax.scatter(gu_filtered.loc[mask, 'n_hits'],
               gu_filtered.loc[mask, 'coverage_frac'],
               alpha=0.5, s=20, c=color, label=cls)
ax.set_xlabel('Number of tBLASTn hits')
ax.set_ylabel('Exon 4-15 coverage fraction')
ax.set_title('Hits vs coverage per gene unit')
ax.axhline(MIN_COVERAGE, color='red', linestyle='--', alpha=0.5)
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(FIG_DIR / 'gene_unit_coverage.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"  Saved: {FIG_DIR / 'gene_unit_coverage.png'}")

## 1c. Write output files

## 1c. Writing output files

In [ ]:
print("\n── 1c. Writing output files ──")

output_fasta = DATA_PROC / 'mroh6_gene_units.fasta'
n_written = gene_units_to_fasta(gu_filtered, output_fasta)
print(f"  Wrote {n_written} gene units to {output_fasta.name}")

# Save metadata
gu_filtered.drop(columns=['sequence'], errors='ignore').to_csv(
    DATA_PROC / 'mroh6_gene_units_table.csv', index=False)
gu_filtered.rename(columns={'gene_unit_id': 'locus_id'}).drop(
    columns=['sequence'], errors='ignore').to_csv(
    DATA_PROC / 'mroh6_loci_table.csv', index=False)
print(f"  Saved gene unit metadata to mroh6_gene_units_table.csv")

## 1d. Align with MAFFT

## 1d. Running MAFFT alignment

In [ ]:
print("\n── 1d. Running MAFFT alignment ──")
aligned_fasta = DATA_PROC / 'mroh6_aligned.fasta'

result = subprocess.run(
    ['mafft', '--auto', '--thread', '-1', str(output_fasta)],
    capture_output=True, text=True
)

if result.returncode == 0:
    with open(aligned_fasta, 'w') as f:
        f.write(result.stdout)
    print(f"  MAFFT alignment complete: {aligned_fasta.name}")
else:
    print(f"  MAFFT error: {result.stderr[:200]}")
    sys.exit(1)

# Alignment QC
alignment = AlignIO.read(aligned_fasta, 'fasta')
n_seqs = len(alignment)
aln_len = alignment.get_alignment_length()
print(f"  Alignment: {n_seqs} sequences x {aln_len} columns")

gap_fracs = np.array([
    alignment[:, i].count('-') / n_seqs for i in range(aln_len)
])
print(f"  Columns with >50% gaps: {(gap_fracs > 0.5).sum()} / {aln_len}")

## Figure 6: Gap distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

axes[0].plot(gap_fracs, color='steelblue', linewidth=0.5)
axes[0].axhline(0.5, color='red', linestyle='--', alpha=0.7, label='50% gap threshold')
axes[0].set_xlabel('Alignment position')
axes[0].set_ylabel('Gap fraction')
axes[0].set_title('Gap distribution (raw alignment)')
axes[0].legend()

# Trim columns with >50% gaps
keep_cols = np.where(gap_fracs <= 0.5)[0]
trimmed_records = []
for record in alignment:
    trimmed_seq = ''.join(str(record.seq)[i] for i in keep_cols)
    trimmed_records.append(SeqRecord(Seq(trimmed_seq), id=record.id, description=''))

trimmed_path = DATA_PROC / 'mroh6_aligned_trimmed.fasta'
SeqIO.write(trimmed_records, trimmed_path, 'fasta')
n_codons = len(keep_cols) // 3
print(f"  Trimmed alignment: {len(trimmed_records)} sequences x {len(keep_cols)} columns ({n_codons} codons)")

# Plot trimmed gaps
trimmed_aln = AlignIO.read(trimmed_path, 'fasta')
gap_fracs_trim = np.array([
    trimmed_aln[:, i].count('-') / len(trimmed_aln)
    for i in range(trimmed_aln.get_alignment_length())
])
axes[1].plot(gap_fracs_trim, color='steelblue', linewidth=0.5)
axes[1].set_xlabel('Alignment position')
axes[1].set_ylabel('Gap fraction')
axes[1].set_title(f'Gap distribution after trimming ({len(keep_cols)} columns)')

plt.tight_layout()
plt.savefig(FIG_DIR / 'alignment_gap_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"  Saved: {FIG_DIR / 'alignment_gap_distribution.png'}")

## Summary

In [ ]:
chr7_out = gu_filtered[gu_filtered['chrom'] == '7']
derived_out = gu_filtered[gu_filtered['chrom'] != '7']

## Summary

In [ ]:
print("\n" + "=" * 70)
print("  DATA PREPARATION SUMMARY (Exon 4-15 Strategy)")

## Summary

In [ ]:
print("=" * 70)
print(f"  Exon 13 anchors:           {len(exon13)}")
print(f"  Combined tBLASTn hits:     {len(tblastn)}")
print(f"  Gene units (all):          {len(gu_all)}")
print(f"  Coverage filter:           >= {MIN_COVERAGE*100:.0f}% of exon 4-15 ({EXON4_15_LEN_NT} nt)")
print(f"  Gene units (filtered):     {len(gu_filtered)}")
print(f"    Chr 7 (ancestral):       {len(chr7_out)}")
print(f"    Macro-derived:           {len(gu_filtered[gu_filtered['chrom_class']=='macro_derived'])}")
print(f"    Micro-derived:           {len(gu_filtered[gu_filtered['chrom_class']=='micro_derived'])}")
print(f"    Sex chromosomes:         {len(gu_filtered[gu_filtered['chrom_class']=='sex_chrom'])}")
print(f"  Alignment:                 {len(trimmed_records)} seqs x {len(keep_cols)} cols ({n_codons} codons)")
print(f"  Expansion ratio:           {len(derived_out)/max(len(chr7_out),1):.1f}x")
print(f"\n  => Proceed to Step 02")

## Summary

In [ ]:
print("=" * 70)